In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
data = pd.read_csv("/content/german_credit_data.csv")
data.head()

,status_account,month_duration,credit_history,purpose,credit_amount,status_savings,years_employment,payment_to_income_ratio,status_and_sex,secondary_obligor,...,collateral,age,other_installment_plans,housing,n_credits,job,n_guarantors,telephone,is_foreign_worker,target
0,< 0 DM,6,critical account/ other credits existing (not ...,radio/television,1169,unknown/ no savings account,>= 7 years,4,male : single,none,...,none,67,none,own,2,skilled employee/ official,1,"yes, registered under the customers name",yes,good
1,0 to < 200 DM,48,existing credits paid back duly till now,radio/television,5951,< 100 DM,1 to < 4 years,2,female : divorced/separated/married,none,...,none,22,none,own,1,skilled employee/ official,1,none,yes,bad
2,no checking account,12,critical account/ other credits existing (not ...,education,2096,< 100 DM,4 to < 7 years,2,male : single,none,...,none,49,none,own,1,unskilled - resident,2,none,yes,good
3,< 0 DM,42,existing credits paid back duly till now,furniture/equipment,7882,< 100 DM,4 to < 7 years,2,male : single,guarantor,...,car,45,none,for free,1,skilled employee/ official,2,none,yes,good
4,< 0 DM,24,delay in paying off in the past,car (new),4870,< 100 DM,1 to < 4 years,3,male : single,none,...,savings agreement/life insurance,53,none,for free,2,skilled employee/ official,2,none,yes,bad


In [3]:
data.info()
data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   status_account           1000 non-null   object
 1   month_duration           1000 non-null   int64 
 2   credit_history           1000 non-null   object
 3   purpose                  1000 non-null   object
 4   credit_amount            1000 non-null   int64 
 5   status_savings           1000 non-null   object
 6   years_employment         1000 non-null   object
 7   payment_to_income_ratio  1000 non-null   int64 
 8   status_and_sex           1000 non-null   object
 9   secondary_obligor        1000 non-null   object
 10  residence_since          1000 non-null   int64 
 11  collateral               1000 non-null   object
 12  age                      1000 non-null   int64 
 13  other_installment_plans  1000 non-null   object
 14  housing                  1000 non-null   

,month_duration,credit_amount,payment_to_income_ratio,residence_since,age,n_credits,n_guarantors
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000


In [4]:
data = data.dropna()
data = data.drop_duplicates()


In [5]:
data['target'] = data["target"].map({"good":1,"bad":0})


In [6]:
X = data.drop("target",axis = 1)
Y = data["target"]

In [7]:
X = pd.get_dummies(X,drop_first=True)

In [8]:
print(Y.value_counts())

target
1    700
0    300
Name: count, dtype: int64


In [9]:
X = pd.get_dummies(X, drop_first=True)

In [10]:
data.columns

Index(['status_account', 'month_duration', 'credit_history', 'purpose',
       'credit_amount', 'status_savings', 'years_employment',
       'payment_to_income_ratio', 'status_and_sex', 'secondary_obligor',
       'residence_since', 'collateral', 'age', 'other_installment_plans',
       'housing', 'n_credits', 'job', 'n_guarantors', 'telephone',
       'is_foreign_worker', 'target'],
      dtype='object')

In [11]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [13]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    class_weight="balanced_subsample",
    random_state=42
)

model.fit(X_train, Y_train)

RandomForestClassifier(class_weight='balanced_subsample', max_depth=12,
                       n_estimators=500, random_state=42)

In [14]:
y_pred = model.predict(X_test)
print(confusion_matrix(Y_test,y_pred))
print(classification_report(Y_test,y_pred))

[[ 29  30]
 [  9 132]]
              precision    recall  f1-score   support

           0       0.76      0.49      0.60        59
           1       0.81      0.94      0.87       141

    accuracy                           0.81       200
   macro avg       0.79      0.71      0.73       200
weighted avg       0.80      0.81      0.79       200



In [15]:
print("Unique prediction:",np.unique(y_pred))

Unique prediction: [0 1]


In [30]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns, "columns.pkl")

['columns.pkl']

In [17]:
feature_columns = X.columns

In [29]:
input_df = pd.DataFrame(0, columns=feature_columns, index=[0])

In [19]:
input_df.loc[0] = np.zeros(len(feature_columns))


In [20]:
input_df.loc[0, "month_duration"] = 12
input_df.loc[0, "credit_amount"] = 20000
input_df.loc[0, "payment_to_income_ratio"] = 30
input_df.loc[0, "age"] = 35
input_df.loc[0, "n_credits"] = 2
input_df.loc[0, "residence_since"] = 4
input_df.loc[0, "status_account_< 0 DM"] = 1

In [21]:
if "housing_own" in input_df.columns:
    input_df.loc[0, "housing_own"] = 1


In [22]:
input_df = input_df.reindex(columns=feature_columns, fill_value=0)
input_df.loc[0, "month_duration"] = 12
input_scaled = scaler.transform(input_df)

In [23]:
prediction = model.predict(input_scaled)
print(f"Prediction: {prediction[0]}")

Prediction: 1


In [25]:
if prediction[0] == 1:
  print("Prediction: Good")
else:
  print("Prediction: Bad")


Prediction: Good


In [26]:
import numpy as np

y_pred = model.predict(X_test)
print(np.unique(y_pred))

[0 1]


In [27]:
print(X.columns.tolist())

['month_duration', 'credit_amount', 'payment_to_income_ratio', 'residence_since', 'age', 'n_credits', 'n_guarantors', 'status_account_< 0 DM', 'status_account_>= 200 DM', 'status_account_no checking account', 'credit_history_critical account/ other credits existing (not at this bank)', 'credit_history_delay in paying off in the past', 'credit_history_existing credits paid back duly till now', 'credit_history_no credits taken/ all credits paid back duly', 'purpose_car (new)', 'purpose_car (used)', 'purpose_domestic appliances', 'purpose_education', 'purpose_furniture/equipment', 'purpose_others', 'purpose_radio/television', 'purpose_repairs', 'purpose_retraining', 'status_savings_500 to < 1000 DM', 'status_savings_< 100 DM', 'status_savings_>= 1000 DM', 'status_savings_unknown/ no savings account', 'years_employment_4 to < 7 years', 'years_employment_< 1 year', 'years_employment_>= 7 years', 'years_employment_unemployed', 'status_and_sex_male : divorced/separated', 'status_and_sex_male 